# cohort ごとの切片の補正 — global AUROC の低下は cohort をまたいだ score のずれか

iterative の stage は、cohort × class の class weight（`weighting=inverse`）を掛けた GroupDRO で学習する。
class weight 付きの cross-entropy を最小化する logit は、真の log odds に cohort ごとの定数 `log(w_pos / w_neg)` を
足したものになる。cohort ごとに陽性率が違うので、この定数も cohort ごとに違い、**cohort をまたいで score の水準がずれる**。
このずれは cohort の中の順位を変えないが、cohort をまたいだ順位（pooled AUROC）を崩す。

[global_training_curves](global_training_curves.ipynb) では、GroupDRO の stage に入ると global AUROC が warmup の水準に戻らないことを見た。
この notebook は、その低下がこの切片のずれで説明できるかを、**学習をやり直さずに**確かめる。

問いは 3 つある。

1. cohort ごとの切片を補正すると、pooled AUROC は baseline に近づくか。
2. モデルが学習した切片のずれは、class weight から予測される量と一致するか。
3. 補正すると、属性群の worst AUROC と、動作点での TPR・FPR の群間差はどう動くか。

split は test。動作点の閾値だけは val で決める。全 run が seed 42 の 1 本である。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET, run_dir, split_csv  # noqa: E402
from analysis.common.predictions import cache_path, load_cache  # noqa: E402
from analysis.common.run_artifacts import read_config  # noqa: E402
from analysis.iterative_probe.groups import demographics_of  # noqa: E402

matplotlib.style.use(STYLE_SHEET)
pd.set_option("display.width", 200)

## 対象 run と定数

run は条件順（fc 1e-3 → fc 1e-2 → stage4+fc 1e-3 → stage4+fc 1e-2）に並べる。checkpoint は各 run の `selected_checkpoint`
（stage02 の best val AUROC）なので、補正に使う cohort と class weight は **stage02** のもの（`cohort02`、`stages/stage02/config.yaml`）である。

cohort は、モデルの入力と同じ metadata（sex・race・ethnicity・frontal/lateral・AP/PA・age）を metadata encoder に通し、
その出力を k-means で 10 個に割ったもの（`cohort.json` の `metadata_kmeans`）。**cohort はモデルの入力から決まる**ので、
モデルは cohort ごとに切片をずらすことを容易に学習できる。

動作点は、val 全体で感度が 0.9 になる閾値とする。スクリーニングを想定した仮の置き方で、学習時の logger が使う argmax（閾値 0.5）は
class weight の掛け方で動くため使わない。

予測 cache は事前に共有 CLI で作る（[README](README.md) の「再実行」）。

In [ ]:
STUDY = "iterative_probe"
ITERATIVE_RUN_IDS = [
    "20260924T112035Z-spatial-lora-iterative-chexpert-fc-s42-b81a",
    "20260924T112045Z-spatial-lora-iterative-chexpert-fc-s42-4602",
    "20260924T112034Z-spatial-lora-iterative-chexpert-stage4-fc-s42-1caa",
    "20260924T112043Z-spatial-lora-iterative-chexpert-stage4-fc-s42-c755",
]
BASELINE_RUN_ID = "20260921T103036Z-resnet-chexpert-s42-5538"
BASELINE_CONDITION = "ResNet ERM"
COHORT_STAGE = "stage02"
SENSITIVITY = 0.9
ATTRIBUTES = ["age", "sex", "race"]
SHUFFLE_SEED = 0

CACHE = ROOT / "analysis" / STUDY / "cache"
RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)

## 入力の読み込み

- score: 予測 cache の logits から、陽性と陰性の logit の差（margin）を取る。softmax の確率は margin の単調変換なので、AUROC はどちらで測っても同じになる。補正は logit の空間で足し引きするので margin を使う。
- cohort: `artifacts/cohorts/cohort02/assignments.parquet` を image でつなぐ。val と test も割り当てを持つ。
- 条件名: run の `config.yaml` から変調範囲と step size を読む。

In [ ]:
def read_margin(run_id: str, split: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    images = pd.read_csv(split_csv("chexpert", split))["image"].to_numpy(dtype=str)
    cache = load_cache(cache_path(CACHE, run_id, split), images)
    return cache["logits"][:, 1] - cache["logits"][:, 0], cache["target"], images


def read_cohorts(run_id: str, split: str, images: np.ndarray) -> np.ndarray:
    path = (
        run_dir(STUDY, run_id)
        / "artifacts"
        / "cohorts"
        / f"cohort{COHORT_STAGE.removeprefix('stage')}"
        / "assignments.parquet"
    )
    assignments = pd.read_parquet(path)
    by_image = assignments[assignments["split"] == split].set_index("image")["group_id"]
    cohorts = by_image.reindex(images)
    if cohorts.isna().any():
        raise ValueError(f"{run_id} の {split} に cohort の無い行がある")
    return cohorts.to_numpy(dtype=int)


def read_condition(run_id: str) -> str:
    config = read_config(run_dir(STUDY, run_id) / "config.yaml")
    modulation = "+".join(config["model"]["net"]["modulation_stages"])
    return f"{modulation} / {float(config['iteration']['group_dro_step_size']):g}"


splits = {}
for run_id in [BASELINE_RUN_ID, *ITERATIVE_RUN_IDS]:
    for split in ["val", "test"]:
        margin, target, images = read_margin(run_id, split)
        splits[(run_id, split)] = {"margin": margin, "target": target, "images": images}
conditions = {run_id: read_condition(run_id) for run_id in ITERATIVE_RUN_IDS} | {BASELINE_RUN_ID: BASELINE_CONDITION}
demographics = demographics_of("test")
{condition: len(splits[(run_id, "test")]["target"]) for run_id, condition in conditions.items()}

## 1. 補正量

3 通りの補正を比べる。どれも cohort ごとの定数を margin から引くだけなので、**cohort の中の順位は変えない**。

- `class-weight offset`: 理論どおりの補正。stage02 の class weight から `log(w_pos / w_neg)` を cohort ごとに引く。
- `val-fitted offset`: val で cohort ごとに切片だけを当てはめた補正（logistic 回帰で切片 1 つを cohort ごとに推定）。cohort ごとの切片のずれを取り除いたときに到達できる上限の目安になる。test は見ない。
- `shuffled offset`: 対照。理論の補正量を cohort 間でランダムに入れ替えたもの。「cohort ごとに何かずらせば pooled AUROC が変わる」のではなく、「class weight どおりにずらすと戻る」ことを確かめる。

In [ ]:
def class_weight_offset(run_id: str) -> np.ndarray:
    config = read_config(run_dir(STUDY, run_id) / "stages" / COHORT_STAGE / "config.yaml")
    weight = np.asarray(config["model"]["loss_fn"]["class_weight"])
    return np.log(weight[:, 1] / weight[:, 0])


def fitted_intercept(margin: np.ndarray, target: np.ndarray) -> float:
    sign = 2 * target - 1

    def negative_log_likelihood(intercept: float) -> float:
        return float(np.mean(np.logaddexp(0, -sign * (margin - intercept))))

    return minimize_scalar(negative_log_likelihood, bounds=(-10, 10), method="bounded").x


def val_fitted_offset(margin: np.ndarray, target: np.ndarray, cohorts: np.ndarray) -> np.ndarray:
    return np.array([fitted_intercept(margin[cohorts == k], target[cohorts == k]) for k in np.unique(cohorts)])


rng = np.random.default_rng(SHUFFLE_SEED)
offsets = {}
offset_rows = []
for run_id in ITERATIVE_RUN_IDS:
    for split in ["val", "test"]:
        part = splits[(run_id, split)]
        part["cohort"] = read_cohorts(run_id, split, part["images"])
    val = splits[(run_id, "val")]
    theory = class_weight_offset(run_id)
    fitted = val_fitted_offset(val["margin"], val["target"], val["cohort"])
    offsets[run_id] = {
        "none": np.zeros_like(theory),
        "class-weight offset": theory,
        "shuffled offset": rng.permutation(theory),
        "val-fitted offset": fitted,
    }
    slope, _ = np.polyfit(theory, fitted, 1)
    offset_rows.append(
        {
            "condition": conditions[run_id],
            "class-weight offset range": theory.max() - theory.min(),
            "val-fitted offset range": fitted.max() - fitted.min(),
            "corr": np.corrcoef(theory, fitted)[0, 1],
            "slope (fitted ~ class weight)": slope,
        }
    )
offset_summary = pd.DataFrame(offset_rows).set_index("condition")
offset_summary.round(3)

`val-fitted offset` は、モデルが実際に学習した cohort ごとの切片のずれを表す。これが class weight から予測される量と
強く相関すれば（`corr`）、モデルは class weight どおりに cohort ごとの切片をずらしたことになる。`slope` が 1 より小さいのは、
理論上のずれの一部だけを学習したことを示す。

cohort ごとに並べて図にする。横軸が class weight からの予測、縦軸が val で当てはめた切片（run ごとに平均を引いて揃える）。

In [ ]:
figure, axes = plt.subplots(1, len(ITERATIVE_RUN_IDS), figsize=(14, 3.6), sharex=True, sharey=True)
for axis, run_id in zip(axes, ITERATIVE_RUN_IDS, strict=True):
    theory = offsets[run_id]["class-weight offset"]
    fitted = offsets[run_id]["val-fitted offset"]
    axis.scatter(theory - theory.mean(), fitted - fitted.mean(), color="#0173B2")
    axis.axline((0, 0), slope=1, color="#9a9a96", linestyle="--", linewidth=0.8)
    axis.set_title(conditions[run_id])
    axis.set_xlabel("log(w_pos / w_neg), centered")
axes[0].set_ylabel("val-fitted intercept, centered")
figure.savefig(FIGURES / "cohort_offset_fitted_vs_class_weight.png", dpi=180, bbox_inches="tight")
plt.show()

## 2. pooled AUROC と cohort 内の AUROC

補正の前後で、test の pooled AUROC と、cohort 内の AUROC（cohort ごとの AUROC を test の件数で重み付けした平均）を並べる。
cohort 内の AUROC は補正で変わらないはずなので、計算の確認を兼ねる。

baseline は cohort を持たないが、**各 iterative run の cohort で baseline の予測を切れば**、cohort 内の AUROC を同じ物差しで比べられる。
iterative が cohort の中の順位まで落としたのか、cohort をまたいだ順位だけを落としたのかを、これで分ける。

In [ ]:
def within_cohort_auroc(margin: np.ndarray, target: np.ndarray, cohorts: np.ndarray) -> float:
    labels = np.unique(cohorts)
    aurocs = [roc_auc_score(target[cohorts == k], margin[cohorts == k]) for k in labels]
    sizes = [int((cohorts == k).sum()) for k in labels]
    return float(np.average(aurocs, weights=sizes))


baseline_test = splits[(BASELINE_RUN_ID, "test")]
ranking_rows = []
for run_id in ITERATIVE_RUN_IDS:
    test = splits[(run_id, "test")]
    if not np.array_equal(test["images"], baseline_test["images"]):
        raise ValueError("baseline と iterative で test の行順が違う")
    for variant, offset in offsets[run_id].items():
        corrected = test["margin"] - offset[test["cohort"]]
        ranking_rows.append(
            {
                "condition": conditions[run_id],
                "variant": variant,
                "pooled_auroc": roc_auc_score(test["target"], corrected),
                "within_cohort_auroc": within_cohort_auroc(corrected, test["target"], test["cohort"]),
                "baseline_within_cohort_auroc": within_cohort_auroc(
                    baseline_test["margin"], baseline_test["target"], test["cohort"]
                ),
            }
        )
ranking = pd.DataFrame(ranking_rows)
baseline_pooled = roc_auc_score(baseline_test["target"], baseline_test["margin"])
ranking["pooled_vs_baseline"] = ranking["pooled_auroc"] - baseline_pooled
ranking.to_csv(RESULTS / "cohort_offset_ranking.csv", index=False)
print(f"baseline pooled AUROC (test): {baseline_pooled:.4f}")
ranking.set_index(["condition", "variant"]).round(4)

補正で baseline との差がどれだけ埋まったかを、補正前の差に対する割合で見る。

In [ ]:
pooled = ranking.pivot_table(index="condition", columns="variant", values="pooled_auroc", sort=False)
gap_before = baseline_pooled - pooled["none"]
recovered = pd.DataFrame(
    {
        "gap to baseline (none)": gap_before,
        "recovered by class-weight offset": (pooled["class-weight offset"] - pooled["none"]) / gap_before,
        "recovered by val-fitted offset": (pooled["val-fitted offset"] - pooled["none"]) / gap_before,
        "change by shuffled offset": pooled["shuffled offset"] - pooled["none"],
    }
)
recovered.round(3)

`val-fitted offset` は、class weight 由来のずれに限らず、cohort ごとの較正のずれなら何でも吸収する。baseline にも同じ補正で
pooled AUROC が上がるなら、iterative だけが得をする比較になる。**baseline の予測を各 iterative run の cohort で切り、val で
cohort ごとに切片を当てはめて test に適用する。** 切片の幅と pooled AUROC の変化が小さければ、cohort ごとのずれは
iterative の stage に入ってから生じたものと言える。

In [ ]:
baseline_val = splits[(BASELINE_RUN_ID, "val")]
control_rows = []
for run_id in ITERATIVE_RUN_IDS:
    val_cohorts = read_cohorts(run_id, "val", baseline_val["images"])
    test_cohorts = read_cohorts(run_id, "test", baseline_test["images"])
    fitted = val_fitted_offset(baseline_val["margin"], baseline_val["target"], val_cohorts)
    corrected = baseline_test["margin"] - fitted[test_cohorts]
    control_rows.append(
        {
            "cohorts of": conditions[run_id],
            "baseline pooled (none)": baseline_pooled,
            "baseline pooled (val-fitted offset)": roc_auc_score(baseline_test["target"], corrected),
            "baseline offset range": fitted.max() - fitted.min(),
            "iterative offset range": np.ptp(offsets[run_id]["val-fitted offset"]),
        }
    )
baseline_control = pd.DataFrame(control_rows).set_index("cohorts of")
baseline_control.to_csv(RESULTS / "cohort_offset_baseline_control.csv")
baseline_control.round(4)

## 3. 属性群の worst AUROC と、動作点での群間差

属性群は [groups.py](groups.py) の切り方（age group 65 歳・sex・race を White / Asian / Black に絞り、3 属性が非欠損の行）を使う。
AUROC は群の中の順位なので、属性群と cohort が一致しない限り、cohort ごとの補正でも値が動く。

動作点は、**補正後の score で** val 全体の感度が 0.9 になる閾値を決め、test の群ごとの TPR・FPR を出す。
deploy では属性を見ずに 1 つの閾値を使う想定である。

In [ ]:
def attribute_metrics(margin: np.ndarray, target: np.ndarray, threshold: float) -> dict[str, float]:
    row = {}
    for attribute in ATTRIBUTES:
        aurocs, tprs, fprs = [], [], []
        for index in demographics.groupby(attribute).groups.values():
            group_margin, group_target = margin[index], target[index]
            predicted = group_margin > threshold
            aurocs.append(roc_auc_score(group_target, group_margin))
            tprs.append(predicted[group_target == 1].mean())
            fprs.append(predicted[group_target == 0].mean())
        row[(attribute, "worst AUROC")] = min(aurocs)
        row[(attribute, "AUROC gap")] = max(aurocs) - min(aurocs)
        row[(attribute, "TPR gap")] = max(tprs) - min(tprs)
        row[(attribute, "FPR gap")] = max(fprs) - min(fprs)
    return row


def sensitivity_threshold(margin: np.ndarray, target: np.ndarray) -> float:
    return float(np.quantile(margin[target == 1], 1 - SENSITIVITY))


baseline_val = splits[(BASELINE_RUN_ID, "val")]
attribute_rows = {
    (BASELINE_CONDITION, "none"): attribute_metrics(
        baseline_test["margin"],
        baseline_test["target"],
        sensitivity_threshold(baseline_val["margin"], baseline_val["target"]),
    )
}
for run_id in ITERATIVE_RUN_IDS:
    val, test = splits[(run_id, "val")], splits[(run_id, "test")]
    for variant, offset in offsets[run_id].items():
        threshold = sensitivity_threshold(val["margin"] - offset[val["cohort"]], val["target"])
        corrected = test["margin"] - offset[test["cohort"]]
        attribute_rows[(conditions[run_id], variant)] = attribute_metrics(corrected, test["target"], threshold)
attribute_table = pd.DataFrame(attribute_rows).T
attribute_table.index.names = ["condition", "variant"]
attribute_table.columns = pd.MultiIndex.from_tuples(attribute_table.columns)
attribute_table.to_csv(RESULTS / "cohort_offset_attribute_metrics.csv")
attribute_table.xs("worst AUROC", axis=1, level=1).round(4)

動作点での群間差（val 全体の感度 0.9 の閾値）。

In [ ]:
attribute_table.loc[:, (slice(None), ["TPR gap", "FPR gap"])].round(3)

## まとめ

数値の正本は `results/cohort_offset_ranking.csv` と `results/cohort_offset_attribute_metrics.csv`。読み取りの記録は
[reports/cohort_logit_correction.md](reports/cohort_logit_correction.md) に残す。

**観察**

- **モデルは class weight どおりに cohort ごとの切片をずらしている。** val で当てはめた cohort ごとの切片は、
  `log(w_pos / w_neg)` と相関 0.96〜0.97 で並ぶ。傾きは 0.54〜0.69 で、理論上のずれの 5〜7 割を学習している。
- **cohort 内の順位は baseline とほぼ同じ。** 同じ cohort で切った cohort 内 AUROC は、iterative が 0.815〜0.823、
  baseline が 0.821〜0.823。stage4+fc は差が 0.005 以内、fc は 0.005〜0.006 低い。
- **pooled AUROC の低下の大半は、cohort をまたいだ切片のずれで説明できる。** baseline（0.8597）との差 0.012〜0.014 のうち、
  class weight どおりの補正で 39〜81%、val で当てはめた補正で 53〜89% が戻る。補正量を cohort 間でシャッフルすると、
  逆に 0.003〜0.031 下がる。
- **cohort ごとの切片のずれは、iterative の stage で生じたものである。** baseline の予測を同じ cohort で切り、val で切片を
  当てはめて補正しても、pooled AUROC は ±0.0004 しか動かない。当てはめた切片の幅も baseline は 0.39〜0.51 で、
  iterative（1.07〜1.50）より小さい。
- **補正すると、属性群の worst AUROC も baseline に近づく。** sex は 0.847〜0.850 → 0.854〜0.859（baseline 0.859）、
  race は 0.820〜0.825 → 0.829〜0.838（baseline 0.832）。
- **動作点での age の群間差は、補正すると baseline より開く。** val の感度 0.9 の閾値で、補正前の age の TPR gap は
  0.094〜0.116、FPR gap は 0.038〜0.071 で、baseline（0.141 / 0.114）より小さい。class weight どおりに補正すると
  0.155〜0.163 / 0.159〜0.183 になり、baseline を超える。

**読めること**

- iterative の stage で global AUROC が下がったのは、cohort の中の順位が崩れたからではなく、cohort ごとの class weight が
  cohort ごとの切片をずらし、cohort をまたいだ順位を崩したためである。
- 動作点での age の公平性が baseline より良く見えるのは、同じ切片のずれが age ごとに閾値をずらしているためである。
  pooled AUROC の低下と age の群間差の縮小は、同じ仕組みの表と裏になっている。

**読めないこと**

- seed 42 の 1 本ずつなので、条件間の差（回復率 39〜81% のばらつきなど）は読まない。
- 補正後も fc では baseline との差が 0.006〜0.008 残る。cohort 内の順位の低下（0.005〜0.006）と大きさが合うが、
  seed 1 本では確かめられない。
